# IgFold on the same 597 nanobodies

Generation procedure for the IgFold arm of the benchmark. Same frozen targets, same scoring
code and same output schema as the SimpleFold and NanoBodyBuilder2 kernels, so the three are
directly mergeable.

Two hazards are handled explicitly:

1. **AntiBERTy checkpoint loading.** PyTorch 2.6 changed `torch.load` to default
   `weights_only=True`, which rejects the released AntiBERTy and IgFold checkpoints. The patch
   below restores the previous default before any model is constructed.
2. **Residue mapping.** The prediction is located inside `hseq` by sequence identity, never by
   index arithmetic. `do_renum=False` keeps the output numbered `1..L` in input order, so
   `seq_offset` must be 0 and `pred_len` must equal `L` for every scored target.

Predicted coordinates are retained and archived with per-file hashes; they are not deleted.

In [ ]:
import os, sys, json, time, glob, shutil, subprocess, hashlib
import numpy as np

WORK, SCRATCH = '/kaggle/working', '/tmp/igf'
os.makedirs(SCRATCH, exist_ok=True)
PRED = f'{SCRATCH}/pred'; os.makedirs(PRED, exist_ok=True)
T0 = time.time()

R = {'model': 'IgFold',
     'settings': {'num_models': 4, 'do_refine': True, 'use_openmm': True, 'do_renum': False,
                  'sequences': 'single heavy chain, key "H"', 'seed': None,
                  'seed_note': 'forward pass is deterministic; OpenMM refinement is a local '
                               'energy minimisation, no stochastic sampling step is seeded'},
     'metric': 'ImmuneBuilder protocol: IMGT regions, N/CA/C/CB atoms',
     'reference_only': {'IgFold_this_benchmark_primary_record':
                        {'cdr1': 1.946, 'cdr2': 1.247, 'cdr3': 2.886, 'framework': 0.827},
                        'note': 'medians of the stored 597-target primary record; compared '
                                'locally after the run, not asserted here'}}

def note(tag):
    R['elapsed_min'] = round((time.time() - T0) / 60, 1)
    R['stage'] = tag
    json.dump(R, open(f'{WORK}/results_igfold.json', 'w'), indent=2, default=str)
    print(f"[{R['elapsed_min']:>6.1f} min] {tag}", flush=True)

def sha_file(path):
    with open(path, 'rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

import torch
assert torch.cuda.device_count() == 2, torch.cuda.device_count()
assert torch.cuda.get_device_capability(0) >= (7, 0), torch.cuda.get_device_capability(0)
R['gpus'] = [torch.cuda.get_device_name(g) for g in range(2)]
print('GPUs visible:', R['gpus'])

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, _ds
TG = json.load(open(_ds[0]))
for t in TG:
    assert t['atoms'] == ['N','CA','C','CB'] and t['numbering'] == 'IMGT'
    for k, col in (('cdr1','CDRH1'), ('cdr2','CDRH2'), ('cdr3','CDRH3')):
        assert t['hseq'][t[k][0]:t[k][1]] == t[col], (t['instance'], k)
assert len({t['instance'] for t in TG}) == len(TG) == 597
R['n_targets'] = len(TG)
R['input_sha256'] = sha_file(_ds[0])
PROBE = TG[0]
print(f"{len(TG)} targets validated; input sha256 {R['input_sha256'][:16]}...")
print('probe target:', PROBE['pdb'] + PROBE['H'])
note('start')

## Install IgFold and the OpenMM refinement backend

IgFold's default refinement is PyRosetta, which needs a licence and is not available here.
`use_openmm=True` selects the released OpenMM path, the same refinement stack this project
already uses for NanoBodyBuilder2.

In [ ]:
def sh(c):
    return subprocess.run(c, shell=True, text=True, capture_output=True)

def must(c, what):
    p = sh(c)
    if p.returncode:
        raise RuntimeError(f'{what} failed:\n{(p.stdout + p.stderr)[-2000:]}')
    return p

must(f'{sys.executable} -m pip install -q igfold', 'install igfold')
must(f'{sys.executable} -m pip install -q openmm pdbfixer biotite', 'install openmm/pdbfixer/biotite')

# PyTorch >= 2.6 defaults torch.load to weights_only=True, which rejects the released
# AntiBERTy and IgFold checkpoints. Restore the previous default before any model loads.
PATCH = '''
import functools, torch
if torch.__version__ >= "2.6":
    torch.load = functools.partial(torch.load, weights_only=False)
'''
open(f'{SCRATCH}/torch_patch.py', 'w').write(PATCH)

probe = f'''
exec(open("{SCRATCH}/torch_patch.py").read())
import igfold, openmm
from igfold import IgFoldRunner
runner = IgFoldRunner()
runner.fold("/tmp/probe.pdb", sequences={{"H": "{PROBE['hseq']}"}},
            do_refine=True, use_openmm=True, do_renum=False)
print("PROBE_OK", igfold.__version__ if hasattr(igfold, "__version__") else "unknown",
      openmm.__version__)
'''
chk = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
if chk.returncode or 'PROBE_OK' not in chk.stdout:
    raise RuntimeError('IgFold end-to-end probe failed:\n' + (chk.stdout + chk.stderr)[-3000:])
R['probe_versions'] = chk.stdout.split('PROBE_OK')[1].strip()
R['probe_atoms'] = sum(1 for l in open('/tmp/probe.pdb') if l.startswith('ATOM'))
print(f"probe OK: {R['probe_versions']}, {R['probe_atoms']} atoms written")

import importlib.metadata
R['versions'] = {}
for pkg in ('igfold', 'antiberty', 'torch', 'openmm', 'pdbfixer', 'biotite', 'numpy'):
    try:
        R['versions'][pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        R['versions'][pkg] = 'not installed'
print(R['versions'])
note('installed')

## Warm the weights single-threaded, then shard across the two GPUs

Both workers would otherwise race to download the same checkpoints and corrupt them.

In [ ]:
w = subprocess.run([sys.executable, '-c',
    f'exec(open("{SCRATCH}/torch_patch.py").read())\n'
    'from igfold import IgFoldRunner; IgFoldRunner()'],
    capture_output=True, text=True)
if w.returncode:
    raise RuntimeError('weight warm-up failed:\n' + (w.stdout + w.stderr)[-2000:])
print('weights warm')

WORKER = '\n'.join([
 f'exec(open("{SCRATCH}/torch_patch.py").read())',
 'import sys, json, os, time',
 'from igfold import IgFoldRunner',
 'shard, nshard, tgt, out = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], sys.argv[4]',
 'TG = json.load(open(tgt))',
 'runner = IgFoldRunner()',
 't0 = time.time(); n = 0',
 'for i, t in enumerate(TG):',
 '    if i % nshard != shard: continue',
 "    tid = t['pdb'] + t['H']",
 "    p = out + '/' + tid + '.pdb'",
 '    if os.path.exists(p): continue',
 '    try:',
 "        runner.fold(p, sequences={'H': t['hseq']},",
 '                    do_refine=True, use_openmm=True, do_renum=False)',
 '        n += 1',
 "        if n % 25 == 0: print('shard %d: %d done, %.0fs' % (shard, n, time.time()-t0), flush=True)",
 '    except Exception as e:',
 "        print('REFINE_FAIL %s %s: %s' % (tid, type(e).__name__, str(e)[:120]), flush=True)",
 '        try:',
 "            runner.fold(p, sequences={'H': t['hseq']}, do_refine=False, do_renum=False)",
 '            n += 1',
 "            open(out + '/' + tid + '.unrefined', 'w').write(type(e).__name__)",
 "            print('UNREFINED %s written' % tid, flush=True)",
 '        except Exception as e2:',
 "            print('FAIL %s %s: %s' % (tid, type(e2).__name__, str(e2)[:120]), flush=True)",
 "print('shard %d finished %d in %.0fs' % (shard, n, time.time()-t0), flush=True)"])
open(f'{SCRATCH}/worker.py', 'w').write(WORKER)
json.dump(TG, open(f'{SCRATCH}/tg.json', 'w'))

t0 = time.time(); procs = []
for g in (0, 1):
    procs.append(subprocess.Popen([sys.executable, f'{SCRATCH}/worker.py', str(g), '2',
                                   f'{SCRATCH}/tg.json', PRED],
                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                 env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}))
    print(f'launched worker {g}', flush=True)
for g, p in enumerate(procs):
    out = p.communicate()[0]
    open(f'{WORK}/worker{g}.log', 'w').write(out)
    R.setdefault('worker_rc', {})[f'w{g}'] = p.returncode
    tail = [l for l in out.split('\n') if 'FAIL' in l or 'finished' in l]
    print(f'--- worker {g} (rc={p.returncode}) ---'); print('\n'.join(tail[-8:]))
    R.setdefault('worker_fail', {})[f'w{g}'] = [l for l in tail if 'FAIL' in l][:20]
R['build_sec'] = round(time.time() - t0, 1)
R['n_predicted'] = len(glob.glob(f'{PRED}/*.pdb'))
R['unrefined'] = sorted(os.path.basename(x)[:-10] for x in glob.glob(f'{PRED}/*.unrefined'))
print(f"\n{R['n_predicted']}/{len(TG)} built in {R['build_sec']/60:.1f} min; "
      f"unrefined fallback: {R['unrefined']}")
note('built')

## Retain the coordinates and the execution record

The 597 predicted structures are archived with per-file hashes, alongside the model checkpoint
hashes, the package environment and the frozen input hash.

In [ ]:
import zipfile, importlib.metadata, platform, inspect

coordinate_files = sorted(glob.glob(f'{PRED}/*.pdb'))
coordinate_manifest = []
with zipfile.ZipFile(f'{WORK}/igfold_coordinates.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for p in coordinate_files:
        name = os.path.basename(p)
        archive.write(p, name)
        coordinate_manifest.append({'path': name, 'sha256': sha_file(p),
                                    'bytes': os.path.getsize(p)})
json.dump(coordinate_manifest, open(f'{WORK}/coordinate_manifest.json', 'w'), indent=2)
print(f'archived {len(coordinate_manifest)} coordinate files')

import igfold, antiberty
checkpoints = {}
for package in (igfold, antiberty):
    base = os.path.dirname(inspect.getfile(package))
    for p in glob.glob(base + '/**/*', recursive=True):
        if os.path.isfile(p) and p.endswith(('.ckpt', '.pt', '.pth', '.bin', '.safetensors')):
            checkpoints[os.path.relpath(p, os.path.dirname(base))] = sha_file(p)
R['checkpoint_sha256'] = checkpoints
print(f'hashed {len(checkpoints)} checkpoint files')
for k in sorted(checkpoints): print('  ', k)

R['run_type'] = 'IgFold generation procedure for the benchmark arm; primary saved scores are not replaced'
R['python'] = sys.version
R['platform'] = platform.platform()
R['packages'] = {d.metadata['Name']: d.version for d in importlib.metadata.distributions()}
open(f'{WORK}/environment.txt', 'w').write(sh(f'{sys.executable} -m pip freeze').stdout)
json.dump(R, open(f'{WORK}/run_provenance.json', 'w'), indent=2, default=str)
note('coordinates retained')

## Score — the same procedure used for SimpleFold and NanoBodyBuilder2

The prediction is mapped onto `hseq` by sequence identity. With `do_renum=False` every target
must give `seq_offset == 0` and `pred_len == L`; both are asserted.

In [ ]:
import biotite.structure.io.pdb as pdb_io

AA3to1 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G',
          'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
          'THR':'T','TRP':'W','TYR':'Y','VAL':'V'}
REGIONS = ['cdr1','cdr2','cdr3','framework']

def kabsch(P, Q):
    Pm, Qm = P.mean(0), Q.mean(0)
    V, _, W = np.linalg.svd((P - Pm).T @ (Q - Qm))
    D = np.diag([1, 1, np.sign(np.linalg.det(V @ W))])
    return V @ D @ W, Pm, Qm

rmsd = lambda a, b: float(np.sqrt(((a - b) ** 2).sum(-1).mean()))

# self-test: a 5 A displaced loop on a perfect framework must read 0.00 / 5.00
_rng = np.random.default_rng(17)
_ref = _rng.normal(size=(20, 4, 3)); _pred = _ref.copy(); _pred[16:] += [5, 0, 0]
_fw = np.arange(20) < 16
_rot, _pm, _qm = kabsch(_pred[_fw], _ref[_fw])
_al = (_pred - _pm) @ _rot + _qm
assert abs(rmsd(_al[_fw], _ref[_fw])) < 1e-9 and abs(rmsd(_al[~_fw], _ref[~_fw]) - 5) < 1e-9
print('scorer self-test passed')

rows = []
for t in TG:
    tid = t['pdb'] + t['H']
    try:
        f = f'{PRED}/{tid}.pdb'
        if not os.path.exists(f):
            R.setdefault('missing', []).append(tid); continue
        a = pdb_io.PDBFile.read(f).get_structure(model=1)
        a = a[np.isin(a.atom_name, ['N','CA','C','CB'])]

        seen, order = {}, []
        for ch, ri, ins, rn in zip(a.chain_id, a.res_id, a.ins_code, a.res_name):
            key = (str(ch), int(ri), str(ins))
            if key not in seen:
                seen[key] = len(order); order.append(AA3to1.get(str(rn), 'X'))
        pseq = ''.join(order)

        off = t['hseq'].find(pseq)
        if off < 0:
            R.setdefault('seq_mismatch', []).append(
                f"{tid}: pred {len(pseq)} vs ref {len(t['hseq'])}"); continue
        Ln = len(t['hseq'])
        pred = np.full((Ln, 4, 3), np.nan)
        ai = {'N':0,'CA':1,'C':2,'CB':3}
        for ch, ri, ins, an, xyz in zip(a.chain_id, a.res_id, a.ins_code, a.atom_name, a.coord):
            pred[seen[(str(ch), int(ri), str(ins))] + off, ai[str(an)]] = xyz
        ref = np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']], float)

        reg = {}
        for k in ('cdr1','cdr2','cdr3'):
            m = np.zeros(Ln, bool); lo, hi = t[k]; m[lo:hi] = True; reg[k] = m
        reg['framework'] = ~(reg['cdr1'] | reg['cdr2'] | reg['cdr3'])

        ok = np.isfinite(ref[..., 0]) & np.isfinite(pred[..., 0])
        if ok[:, 1].sum() < 60:
            R.setdefault('skipped', []).append(f'{tid}:coverage'); continue

        row = {'instance': t['instance'], 'tid': tid, 'L': Ln,
               'cdr3_len': int(t['cdr3'][1] - t['cdr3'][0]), 'n_atoms_matched': int(ok.sum()),
               'seq_offset': int(off), 'pred_len': len(pseq),
               'method': t['method'], 'holo': t['holo'], 'ab_split': t['ab_split'],
               'resolution': t['resolution']}
        for mode, fitmask in (('chain', np.ones(Ln, bool)), ('fw', reg['framework'])):
            sel = ok & fitmask[:, None]
            if sel.sum() < 60: continue
            Rot, Pm, Qm = kabsch(pred[sel], ref[sel])
            al = np.full_like(pred, np.nan)
            al[ok] = (pred[ok] - Pm) @ Rot + Qm
            for rn in REGIONS:
                s2 = ok & reg[rn][:, None]
                row[f'{mode}_{rn}'] = round(rmsd(al[s2], ref[s2]), 3) if s2.sum() >= 3 else None
        rows.append(row)
    except Exception as e:
        R.setdefault('score_err', []).append(f'{tid}:{type(e).__name__}:{str(e)[:90]}')

R['rows'] = rows
print(f'scored {len(rows)} / {len(TG)}')
for k in ('missing','seq_mismatch','skipped','score_err'):
    if R.get(k): print(f'  {k}: {len(R[k])}  e.g. {R[k][:3]}')
R['n_scored'] = len(rows)
bad = [r['tid'] for r in rows if r['seq_offset'] != 0 or r['pred_len'] != r['L']]
R['offset_or_length_violations'] = bad
print(f'residue-preservation violations: {len(bad)}' + (f' {bad[:5]}' if bad else ' (none)'))

import csv
cols = (['instance','tid','L','cdr3_len','n_atoms_matched','seq_offset','pred_len',
         'method','holo','ab_split','resolution']
        + [f'{m}_{r}' for m in ('chain','fw') for r in REGIONS])
json.dump(rows, open(f'{WORK}/per_target_scores_igfold.json','w'), indent=1)
with open(f'{WORK}/per_target_scores_igfold.csv','w',newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=cols, extrasaction='ignore', restval='')
    w.writeheader(); w.writerows(rows)
note('scored')

## Results

In [ ]:
def summarise(rr):
    rng = np.random.default_rng(0)
    out = {'n': len(rr)}
    for mode in ('chain','fw'):
        d = {}
        for rn in REGIONS:
            v = np.array([r[f'{mode}_{rn}'] for r in rr if r.get(f'{mode}_{rn}') is not None])
            if len(v) >= 5:
                bs = np.median(v[rng.integers(0, len(v), (2000, len(v)))], axis=1)
                d[rn] = {'median': round(float(np.median(v)), 3), 'mean': round(float(v.mean()), 3),
                         'ci95': [round(float(np.percentile(bs, 2.5)), 3),
                                  round(float(np.percentile(bs, 97.5)), 3)], 'n': int(len(v))}
        out[f'{mode}_aligned'] = d
    return out

S = summarise(rows)
ref = R['reference_only']['IgFold_this_benchmark_primary_record']
print(f"n={S['n']}\n")
print(f"{'region':<11}{'this run':>10}{'95% CI':>18}{'primary record':>16}{'delta':>9}")
for rn in REGIONS:
    e = S['chain_aligned'][rn]
    ci = '[' + str(e['ci95'][0]) + ', ' + str(e['ci95'][1]) + ']'
    print(f"{rn:<11}{e['median']:>10.3f}{ci:>18}{ref[rn]:>16.3f}{e['median']-ref[rn]:>+9.3f}")
print('\nframework-aligned:',
      {rn: S['fw_aligned'][rn]['median'] for rn in REGIONS})
print('\nPer-target agreement with the primary record is computed in the repository, '
      'not here; this kernel only reports its own scores.')
R['summary'] = S
note('done')
print('\nwrote results_igfold.json, per_target_scores_igfold.{json,csv}, '
      'igfold_coordinates.zip, coordinate_manifest.json, run_provenance.json, environment.txt')